# Pipeline 3 — Multi-Stage RAG (Iterative Dense Retrieval + Query Refinement)

**Method:** Two-stage iterative retrieval where the LLM refines the query between stages.

```
Question ──→ Stage 1: Dense Retrieve ──→ LLM refines query ──→ Stage 2: Dense Retrieve
                                                                       ↓
                                                              Merge + Dedup → LLM Answer
```

**Features:**
- 14-key Groq API carousel with rate-limit tracking
- Parallel sample processing (14 workers)
- Each sample makes 2 LLM calls (refine query + final answer) — key rotation handles the load

## Step 1 — Install Dependencies

In [ ]:
!pip install -q datasets numpy sentence-transformers chromadb groq tqdm ragas langchain-groq langchain-core

## Step 1b — Mount Google Drive & Set Paths

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# --- Path to your project folder on Google Drive ---
BASE_DIR     = '/content/drive/MyDrive/HotPotQA-Coding-Trials'
API_KEYS_CSV = f'{BASE_DIR}/api_keys.csv'
RESULTS_DIR  = f'{BASE_DIR}/results/embedding_models/e5-base-v2'

import os
os.makedirs(RESULTS_DIR, exist_ok=True)
print(f"Drive mounted. Project dir: {BASE_DIR}")

## Step 2 — Imports and Configuration

In [ ]:
import os, json, re, time, random, collections, string, csv, threading
import numpy as np
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm.notebook import tqdm
from datasets import load_dataset
from sentence_transformers import SentenceTransformer
import chromadb
from groq import Groq
from datetime import datetime

GROQ_MODEL    = "llama-3.3-70b-versatile"
DENSE_MODEL   = "intfloat/e5-base-v2"
TOP_K         = 4
N_STAGES      = 2
N_SAMPLES     = 500
SEED          = 42
MAX_WORKERS   = 14
PIPELINE_NAME = "MultiStageRAG_e5-base-v2"
# API_KEYS_CSV and RESULTS_DIR are set in the Drive mount cell above

print(f"Pipeline : {PIPELINE_NAME}")
print(f"LLM      : {GROQ_MODEL} | Dense: {DENSE_MODEL}")
print(f"Stages   : {N_STAGES} | Workers: {MAX_WORKERS} | Top-K: {TOP_K}")

## Step 3 — API Key Manager (14-Key Carousel)

In [ ]:
class APIKeyManager:
    BUFFERS = {'rpm': 25, 'rpd': 900, 'tpm': 10_000, 'tpd': 90_000}

    def __init__(self, csv_path):
        self.keys = self._load_keys(csv_path)
        self._lock = threading.Lock()
        self._idx = 0
        self._usage = {k: {'min_req': [], 'day_req': [], 'min_tok': [], 'day_tok': []} for k in self.keys}
        print(f"Loaded {len(self.keys)} API keys")

    @staticmethod
    def _load_keys(csv_path):
        keys = []
        with open(csv_path, 'r') as f:
            for row in csv.DictReader(f):
                k = row.get('API_KEY', '').strip()
                if k: keys.append(k)
        if not keys: raise ValueError(f"No keys in {csv_path}")
        return keys

    def _clean(self, key):
        now = time.time()
        u = self._usage[key]
        u['min_req'] = [t for t in u['min_req'] if now - t < 60]
        u['day_req'] = [t for t in u['day_req'] if now - t < 86400]
        u['min_tok'] = [(t, n) for t, n in u['min_tok'] if now - t < 60]
        u['day_tok'] = [(t, n) for t, n in u['day_tok'] if now - t < 86400]

    def _is_available(self, key):
        self._clean(key)
        u = self._usage[key]
        return (len(u['min_req']) < self.BUFFERS['rpm']
                and len(u['day_req']) < self.BUFFERS['rpd']
                and sum(n for _, n in u['min_tok']) < self.BUFFERS['tpm']
                and sum(n for _, n in u['day_tok']) < self.BUFFERS['tpd'])

    def get_key(self):
        with self._lock:
            for _ in range(len(self.keys)):
                key = self.keys[self._idx]
                self._idx = (self._idx + 1) % len(self.keys)
                if self._is_available(key): return key
            return self._wait_and_get()

    def _wait_and_get(self):
        min_wait = 60
        for key in self.keys:
            reqs = self._usage[key]['min_req']
            if reqs: min_wait = min(min_wait, max(0, 60 - (time.time() - min(reqs))))
        print(f"  ⏳ All keys busy — waiting {min_wait:.1f}s...")
        time.sleep(min_wait + 1)
        for _ in range(len(self.keys)):
            key = self.keys[self._idx]
            if self._is_available(key): return key
            self._idx = (self._idx + 1) % len(self.keys)
        return self.keys[self._idx]

    def record(self, key, tokens=0):
        with self._lock:
            now = time.time()
            self._usage[key]['min_req'].append(now)
            self._usage[key]['day_req'].append(now)
            if tokens > 0:
                self._usage[key]['min_tok'].append((now, tokens))
                self._usage[key]['day_tok'].append((now, tokens))

    def mark_exhausted(self, key):
        with self._lock:
            self._usage[key]['min_req'].extend([time.time()] * self.BUFFERS['rpm'])
            self._idx = (self._idx + 1) % len(self.keys)

    def status(self):
        with self._lock:
            for i, key in enumerate(self.keys):
                self._clean(key); u = self._usage[key]
                print(f"  Key {i+1:2d}: {len(u['min_req']):3d}/{self.BUFFERS['rpm']} RPM  "
                      f"{len(u['day_req']):4d}/{self.BUFFERS['rpd']} RPD")

key_manager = APIKeyManager(API_KEYS_CSV)

## Step 4 — Load HotPotQA Data

In [ ]:
ds = load_dataset("hotpotqa/hotpot_qa", "distractor", split="validation")
if N_SAMPLES is not None:
    random.seed(SEED)
    samples = ds.select(random.sample(range(len(ds)), min(N_SAMPLES, len(ds))))
else:
    samples = ds
print(f"Loaded {len(samples)} samples")

## Step 5 — Data Processing

In [ ]:
def process_context(context):
    candidates = []
    for title, sentences in zip(context['title'], context['sentences']):
        for i, sent in enumerate(sentences):
            candidates.append({'title': title, 'sent_id': i, 'text': sent})
    return candidates

def format_gold_supporting_facts(sf):
    return [{'title': t, 'sent_id': s} for t, s in zip(sf['title'], sf['sent_id'])]

## Step 6 — Dense Retriever (ChromaDB / intfloat/e5-base-v2)

In [ ]:
embed_model = SentenceTransformer(DENSE_MODEL)
print(f"Dense model loaded: {DENSE_MODEL}")

_chroma_client = chromadb.EphemeralClient()
_chroma_lock = threading.Lock()

def dense_retrieve(query, candidates, k=5):
    if not candidates:
        return []
    texts  = ["passage: " + c['text'] for c in candidates]
    cand_emb = embed_model.encode(texts, convert_to_numpy=True, show_progress_bar=False)
    q_emb    = embed_model.encode(["query: " + query], convert_to_numpy=True, show_progress_bar=False)[0]
    # normalise for cosine similarity
    norms    = np.linalg.norm(cand_emb, axis=1, keepdims=True)
    cand_emb = cand_emb / np.maximum(norms, 1e-10)
    q_norm   = np.linalg.norm(q_emb)
    q_emb    = q_emb / max(q_norm, 1e-10)

    col_name = f"tmp_{threading.get_ident()}"
    with _chroma_lock:
        collection = _chroma_client.get_or_create_collection(
            name=col_name, metadata={"hnsw:space": "cosine"})
        ids = [str(i) for i in range(len(candidates))]
        collection.add(ids=ids, embeddings=cand_emb.tolist())
        results = collection.query(
            query_embeddings=[q_emb.tolist()], n_results=min(k, len(candidates)))
        _chroma_client.delete_collection(col_name)
    retrieved = []
    for idx_str, distance in zip(results['ids'][0], results['distances'][0]):
        i = int(idx_str)
        retrieved.append(dict(**candidates[i], score=float(1.0 / (1.0 + distance))))
    return retrieved

## Step 7 — LLM Client (Multi-Key Groq)

Includes both `generate()` (for final answer, 512 tokens) and `generate_short()` (for query refinement, 128 tokens).

In [ ]:
class GroqClient:
    def __init__(self, model_name, km):
        self.model_name = model_name
        self.km = km
        self._clients = {}
        self._clock = threading.Lock()

    def _client_for(self, api_key):
        with self._clock:
            if api_key not in self._clients:
                self._clients[api_key] = Groq(api_key=api_key)
            return self._clients[api_key]

    def _call_with_usage(self, prompt, system_prompt, max_tokens, temperature, max_retries=3):
        for _ in range(max_retries):
            api_key = self.km.get_key()
            client = self._client_for(api_key)
            try:
                msgs = []
                if system_prompt:
                    msgs.append({"role": "system", "content": system_prompt})
                msgs.append({"role": "user", "content": prompt})
                resp = client.chat.completions.create(
                    model=self.model_name,
                    messages=msgs,
                    max_tokens=max_tokens,
                    temperature=temperature,
                )
                usage = {
                    "input_tokens": int(getattr(resp.usage, "prompt_tokens", 0) or 0),
                    "output_tokens": int(getattr(resp.usage, "completion_tokens", 0) or 0),
                    "total_tokens": int(getattr(resp.usage, "total_tokens", 0) or 0),
                }
                self.km.record(api_key, usage["total_tokens"])
                return {
                    "text": resp.choices[0].message.content,
                    "usage": usage,
                }
            except Exception as e:
                if '429' in str(e) or 'rate_limit' in str(e).lower():
                    self.km.mark_exhausted(api_key)
                    continue
                print(f"  LLM error: {e}")
                return {
                    "text": "",
                    "usage": {"input_tokens": 0, "output_tokens": 0, "total_tokens": 0},
                }
        return {
            "text": "",
            "usage": {"input_tokens": 0, "output_tokens": 0, "total_tokens": 0},
        }

    def generate(self, prompt, system_prompt=None):
        return self._call_with_usage(prompt, system_prompt, max_tokens=512, temperature=0.1)["text"]

    def generate_short(self, prompt, system_prompt=None):
        result = self._call_with_usage(prompt, system_prompt, max_tokens=128, temperature=0.0)["text"]
        return result if result else prompt

    def generate_short_with_usage(self, prompt, system_prompt=None):
        out = self._call_with_usage(prompt, system_prompt, max_tokens=128, temperature=0.0)
        text = out["text"] if out["text"] else prompt
        return text, out["usage"]

    @staticmethod
    def parse_json_output(text):
        try:
            return json.loads(text)
        except:
            pass
        m = re.search(r"```json\s*(.*?)\s*```", text, re.DOTALL)
        if m:
            try:
                return json.loads(m.group(1))
            except:
                pass
        m = re.search(r"(\{.*\})", text, re.DOTALL)
        if m:
            try:
                return json.loads(m.group(1))
            except:
                pass
        return {"answer": "JSON_PARSE_ERROR", "supporting_facts": [], "raw_output": text}

    def predict(self, prompt):
        out = self._call_with_usage(prompt, None, max_tokens=512, temperature=0.1)
        return self.parse_json_output(out["text"])

    def predict_with_usage(self, prompt):
        out = self._call_with_usage(prompt, None, max_tokens=512, temperature=0.1)
        parsed = self.parse_json_output(out["text"])
        return parsed, out["usage"]

llm = GroqClient(GROQ_MODEL, key_manager)
print("LLM client ready (multi-key, with generate_short).")

## Step 8 — Prompt Construction (Answer + Query Refinement)

In [ ]:
def construct_prompt(question, retrieved_sentences):
    context_str = ""
    for i, item in enumerate(retrieved_sentences, 1):
        context_str += (f"[{i}] Title: {item['title']}\n"
                        f"    Sentence ID: {item['sent_id']}\n"
                        f"    Text: {item['text']}\n\n")
    return f"""You are a helpful assistant for Question Answering.
Answer the following question based ONLY on the provided context sentences.
You must also identify which sentences support your answer.

Context:
{context_str}

Question: {question}

Instructions:
1. Provide a short, concise answer.
2. List the supporting facts as title + sent_id pairs.
3. Use EXACT titles and sent_ids from the context.
4. Output valid JSON only.

Format:
{{{{
  "answer": "...",
  "supporting_facts": [{{{{"title": "...", "sent_id": N}}}}, ...]
}}}}"""


def construct_query_refinement_prompt(question, retrieved_context):
    ctx = ""
    for i, item in enumerate(retrieved_context, 1):
        ctx += f"[{i}] {item['title']}: {item['text']}\n"
    return f"""Based on the original question and the retrieved context below, generate a refined search query
that would help find additional relevant information to answer the question.
Focus on identifying missing information or related entities not yet covered.

Original question: {question}

Retrieved context:
{ctx}

Output ONLY the refined query (one line, no explanation):"""

## Step 9 — Evaluator

In [ ]:
def normalize_answer(s):
    s = str(s) if s is not None else ""
    s = s.lower()
    s = re.sub(r'\b(a|an|the)\b', ' ', s)
    s = ''.join(ch for ch in s if ch not in string.punctuation)
    return ' '.join(s.split())

def answer_f1(pred, gold):
    pt, gt = normalize_answer(pred).split(), normalize_answer(gold).split()
    common = collections.Counter(pt) & collections.Counter(gt)
    ns = sum(common.values())
    if ns == 0:
        return 0.0
    p, r = ns / len(pt), ns / len(gt)
    return (2 * p * r) / (p + r)

def answer_em(pred, gold):
    return float(normalize_answer(pred) == normalize_answer(gold))

def sp_metrics(pred_sp, gold_sp):
    def to_set(lst):
        return {(x['title'], x['sent_id']) if isinstance(x, dict) else (x[0], x[1]) for x in lst}
    ps, gs = to_set(pred_sp), to_set(gold_sp)
    tp = len(ps & gs)
    prec = tp / len(ps) if ps else 0.0
    rec  = tp / len(gs) if gs else 0.0
    f1   = 2 * prec * rec / (prec + rec) if (prec + rec) > 0 else 0.0
    em   = 1.0 if ps == gs and len(gs) > 0 else 0.0
    return {'sp_em': em, 'sp_f1': f1, 'sp_prec': prec, 'sp_recall': rec}

def reciprocal_rank_first_hit(retrieved_context, gold_sp):
    gold_set = {
        (str(item['title']).strip().lower(), int(item['sent_id']))
        for item in gold_sp
        if isinstance(item, dict) and 'title' in item and 'sent_id' in item
    }
    if not gold_set:
        return 0.0, None

    for rank, item in enumerate(retrieved_context, 1):
        key = (str(item.get('title', '')).strip().lower(), int(item.get('sent_id', -1)))
        if key in gold_set:
            return 1.0 / rank, rank

    return 0.0, None

## Step 10 — Run Multi-Stage RAG (Parallel)

Each worker processes one sample through both stages:
1. Stage 1: Dense retrieve with original question
2. LLM refines the query based on stage-1 evidence
3. Stage 2: Dense retrieve with refined query → merge & dedup
4. Final LLM answer from merged context

In [ ]:
def process_sample(sample):
    t0 = time.time()
    question   = sample['question']
    candidates = process_context(sample['context'])
    stage_log  = []

    total_input_tokens = 0
    total_output_tokens = 0
    total_tokens = 0
    retrieval_calls = 0

    # Stage 1
    current_query = question
    all_retrieved = []
    s1 = dense_retrieve(current_query, candidates, k=TOP_K)
    retrieval_calls += 1
    all_retrieved.extend(s1)
    stage_log.append({'stage': 1, 'query': current_query,
                      'results': [(r['title'], r['sent_id']) for r in s1]})

    # Stages 2..N
    for stage in range(2, N_STAGES + 1):
        ref_prompt = construct_query_refinement_prompt(question, all_retrieved)
        refined_text, ref_usage = llm.generate_short_with_usage(ref_prompt)
        total_input_tokens += int(ref_usage.get('input_tokens', 0) or 0)
        total_output_tokens += int(ref_usage.get('output_tokens', 0) or 0)
        total_tokens += int(ref_usage.get('total_tokens', 0) or 0)

        refined = refined_text.strip().split('\n')[0].strip()
        if not refined or len(refined) < 5:
            refined = question

        sn = dense_retrieve(refined, candidates, k=TOP_K)
        retrieval_calls += 1
        existing = {(r['title'], r['sent_id']) for r in all_retrieved}
        for r in sn:
            if (r['title'], r['sent_id']) not in existing:
                all_retrieved.append(r)
                existing.add((r['title'], r['sent_id']))
        stage_log.append({'stage': stage, 'query': refined,
                          'n_results': len(sn)})
        current_query = refined

    all_retrieved.sort(key=lambda x: x.get('score', 0), reverse=True)
    final_ctx = all_retrieved[:TOP_K]

    prompt = construct_prompt(question, final_ctx)
    resp, ans_usage = llm.predict_with_usage(prompt)
    total_input_tokens += int(ans_usage.get('input_tokens', 0) or 0)
    total_output_tokens += int(ans_usage.get('output_tokens', 0) or 0)
    total_tokens += int(ans_usage.get('total_tokens', 0) or 0)

    elapsed = time.time() - t0
    gold_sp = format_gold_supporting_facts(sample['supporting_facts'])

    # Groq Llama-3.3-70b: input=$0.59/1M, output=$0.79/1M → output is 1.34x more expensive
    cost_proxy = total_input_tokens + (1.34 * total_output_tokens)
    rr, first_hit_rank = reciprocal_rank_first_hit(final_ctx, gold_sp)

    return {
        'pred': {'answer': resp.get('answer', ''), 'supporting_facts': resp.get('supporting_facts', [])},
        'gold': {'answer': sample['answer'], 'supporting_facts': gold_sp},
        'detail': {
            'id': sample['id'], 'question': question,
            'gold_answer': sample['answer'], 'gold_sp': gold_sp,
            'pred_answer': resp.get('answer', ''), 'pred_sp': resp.get('supporting_facts', []),
            'retrieved_context': final_ctx,
            'all_retrieved': [(r['title'], r['sent_id']) for r in all_retrieved],
            'stage_log': stage_log, 'time_taken': elapsed,
            'pipeline': PIPELINE_NAME, 'raw_prediction': resp,
            'retrieval_calls': retrieval_calls,
            'token_usage': {
                'input_tokens': total_input_tokens,
                'output_tokens': total_output_tokens,
                'total_tokens': total_tokens,
            },
            'cost_proxy': cost_proxy,
            'first_hit_rank': first_hit_rank,
            'reciprocal_rank': rr,
        }
    }

experiment_start = datetime.now()
print(f"Starting {PIPELINE_NAME} at {experiment_start.strftime('%H:%M:%S')}")
print(f"{len(samples)} samples × {MAX_WORKERS} workers × {N_STAGES} stages\n")

results = []
with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    fmap = {executor.submit(process_sample, s): i for i, s in enumerate(samples)}
    for future in tqdm(as_completed(fmap), total=len(fmap), desc=PIPELINE_NAME):
        try:
            r = future.result(); r['idx'] = fmap[future]; results.append(r)
        except Exception as e:
            print(f"  Error on sample {fmap[future]}: {e}")

results.sort(key=lambda x: x['idx'])
predictions = [r['pred']   for r in results]
golds       = [r['gold']   for r in results]
details     = [r['detail'] for r in results]

experiment_end = datetime.now()
total_time = (experiment_end - experiment_start).total_seconds()
print(f"\nDone in {total_time:.1f}s ({total_time/len(samples):.2f}s/sample)")

## Step 11 — Evaluation

In [ ]:
metrics = {'em':0,'f1':0,'sp_em':0,'sp_f1':0,'sp_prec':0,'sp_recall':0,'joint_em':0,'joint_f1':0}
for pred, gold in zip(predictions, golds):
    em = answer_em(pred['answer'], gold['answer'])
    f1 = answer_f1(pred['answer'], gold['answer'])
    sp = sp_metrics(pred['supporting_facts'], gold['supporting_facts'])
    metrics['em'] += em
    metrics['f1'] += f1
    metrics['sp_em'] += sp['sp_em']
    metrics['sp_f1'] += sp['sp_f1']
    metrics['sp_prec'] += sp['sp_prec']
    metrics['sp_recall'] += sp['sp_recall']
    metrics['joint_em'] += em * sp['sp_em']
    metrics['joint_f1'] += f1 * sp['sp_f1']

n = len(predictions)
for k in metrics:
    metrics[k] /= n

latencies = [float(d.get('time_taken', 0.0)) for d in details]
rrs = [float(d.get('reciprocal_rank', 0.0)) for d in details]
costs = [float(d.get('cost_proxy', 0.0)) for d in details]

metrics['mrr'] = float(np.mean(rrs)) if rrs else 0.0
metrics['latency_mean_s'] = float(np.mean(latencies)) if latencies else 0.0
metrics['latency_median_s'] = float(np.median(latencies)) if latencies else 0.0
metrics['latency_p95_s'] = float(np.percentile(latencies, 95)) if latencies else 0.0
metrics['cost_proxy_mean'] = float(np.mean(costs)) if costs else 0.0
metrics['cost_proxy_total'] = float(np.sum(costs)) if costs else 0.0

print(f"{'='*60}")
print(f"  {PIPELINE_NAME} Results ({n} samples, {N_STAGES} stages)")
print(f"{'='*60}")
for k, v in metrics.items():
    print(f"  {k:20s}: {v:.4f}")
print(f"{'='*60}")

## RAGAS Faithfulness Evaluation

Measures what fraction of the generated answer's claims can be inferred from the retrieved context.

In [ ]:
# --- RAGAS Faithfulness (parallel-wave, all keys) ---
print("\nRunning RAGAS Faithfulness evaluation (parallel waves)...")

import warnings, time
from concurrent.futures import ThreadPoolExecutor, as_completed
warnings.filterwarnings('ignore', category=DeprecationWarning, module='ragas')

from ragas import evaluate as ragas_evaluate
from ragas.metrics import Faithfulness
from ragas.run_config import RunConfig
from langchain_groq import ChatGroq
from ragas.llms import LangchainLLMWrapper
from datasets import Dataset

faithfulness_metric = Faithfulness()

# --- tunables ----------------------------------------------------------
WAVE_COOLDOWN  = 15   # seconds between waves (let TPM window breathe)
SAMPLE_TIMEOUT = 180  # per-sample RAGAS timeout
MAX_RETRIES    = 3    # retries per sample on 429
# -----------------------------------------------------------------------

ragas_run_cfg = RunConfig(timeout=SAMPLE_TIMEOUT, max_retries=3, max_wait=30)

# Build per-sample dicts once
sample_dicts = []
for d in details:
    ctx = [item["text"] for item in d.get("retrieved_context", [])
           if isinstance(item, dict) and "text" in item]
    sample_dicts.append({
        "question": d.get("question", ""),
        "answer":   d.get("pred_answer", ""),
        "contexts": ctx if ctx else [""],
    })

n_keys    = len(key_manager.keys)
n_samples = len(sample_dicts)
print(f"{n_samples} samples | {n_keys} keys | ~{(n_samples + n_keys - 1) // n_keys} waves")


def evaluate_one(sample_idx, api_key, key_no):
    """Evaluate a single sample with the given API key. Returns (idx, score)."""
    ds = Dataset.from_dict({
        "question": [sample_dicts[sample_idx]["question"]],
        "answer":   [sample_dicts[sample_idx]["answer"]],
        "contexts": [sample_dicts[sample_idx]["contexts"]],
    })
    for attempt in range(MAX_RETRIES):
        try:
            llm = ChatGroq(
                model="llama-3.3-70b-versatile",
                groq_api_key=api_key,
                temperature=0,
            )
            res = ragas_evaluate(
                ds,
                metrics=[Faithfulness()],
                llm=LangchainLLMWrapper(llm),
                run_config=ragas_run_cfg,
            )
            score = res.to_pandas()["faithfulness"].iloc[0]
            return (sample_idx, score)
        except Exception as e:
            err = str(e)
            if "429" in err or "rate_limit" in err.lower():
                key_manager.mark_exhausted(api_key)
                api_key = key_manager.get_key()
                key_no  = key_manager.keys.index(api_key) + 1
                wait = 5 * (attempt + 1)
                print(f"    [sample {sample_idx+1}] 429 → rotated to key #{key_no}, "
                      f"retry {attempt+1}/{MAX_RETRIES} after {wait}s")
                time.sleep(wait)
            else:
                print(f"    [sample {sample_idx+1}] error: {e}")
                return (sample_idx, None)
    return (sample_idx, None)


# --- Run in waves of n_keys -------------------------------------------
scores = [None] * n_samples
wave = 0
i = 0  # next sample index to schedule

while i < n_samples:
    wave += 1
    wave_end = min(i + n_keys, n_samples)
    wave_size = wave_end - i
    print(f"\n  Wave {wave}: samples {i+1}-{wave_end} ({wave_size} in parallel)")

    futures = {}
    with ThreadPoolExecutor(max_workers=wave_size) as pool:
        for j in range(wave_size):
            sample_idx = i + j
            key_idx    = j % n_keys
            api_key    = key_manager.keys[key_idx]
            key_no     = key_idx + 1
            fut = pool.submit(evaluate_one, sample_idx, api_key, key_no)
            futures[fut] = (sample_idx, key_no)

        for fut in as_completed(futures):
            sidx, kno = futures[fut]
            idx, score = fut.result()
            tag = f"{score:.4f}" if score is not None and score == score else "FAIL"
            print(f"    sample {idx+1:2d}  key #{kno:2d}  → {tag}")
            scores[idx] = score

    i = wave_end
    if i < n_samples:
        print(f"  Cooling down {WAVE_COOLDOWN}s before next wave...")
        time.sleep(WAVE_COOLDOWN)

# --- Aggregate ---------------------------------------------------------
valid = [s for s in scores if s is not None and s == s]
avg_faithfulness = sum(valid) / len(valid) if valid else 0.0
metrics['faithfulness'] = avg_faithfulness

print(f"\n{'='*50}")
print(f"  RAGAS Faithfulness: {avg_faithfulness:.4f}")
print(f"  (computed on {len(valid)}/{n_samples} samples)")
print(f"{'='*50}")

## Step 12 — Save Results

In [ ]:
os.makedirs(RESULTS_DIR, exist_ok=True)
ts = experiment_start.strftime('%Y%m%d_%H%M%S')
out_file = f'{RESULTS_DIR}/multi_stage_{ts}_results.json'

latencies = [float(d.get('time_taken', 0.0)) for d in details]

with open(out_file, 'w') as f:
    json.dump({'args': {'pipeline': PIPELINE_NAME, 'model': GROQ_MODEL, 'dense_model': DENSE_MODEL,
               'top_k': TOP_K, 'n_stages': N_STAGES, 'n_samples': N_SAMPLES, 'max_workers': MAX_WORKERS},
               'metrics': metrics,
               'timing': {'total_s': total_time,
                          'avg_s': total_time/n,
                          'latency_mean_s': float(np.mean(latencies)) if latencies else 0.0,
                          'latency_median_s': float(np.median(latencies)) if latencies else 0.0,
                          'latency_p95_s': float(np.percentile(latencies, 95)) if latencies else 0.0},
               'details': details,
               'faithfulness_per_sample': scores if 'scores' in globals() else []}, f, indent=2)
print(f"Saved → {out_file}")

## Step — Export Metrics CSV


In [ ]:
# --- Export metrics to CSV ---------------------------------------------------
import csv, os

CSV_PATH = f"{BASE_DIR}/results/final_thesis_run.csv"
os.makedirs(os.path.dirname(CSV_PATH), exist_ok=True)

# Derive a unique pipeline label  (e.g. "VanillaRAG_FAISS", "GraphRAG_Qdrant")
_store = RESULTS_DIR.rstrip("/").split("/")[-1]            # faiss / lancedb / chromadb / qdrant
_label = PIPELINE_NAME
if _store.lower() not in _label.lower():                   # VanillaRAG has no suffix
    _label = f"{PIPELINE_NAME}_{_store.upper()}"

row = {
    "Pipeline":            _label,
    "em":                  round(metrics.get("em", 0), 4),
    "f1":                  round(metrics.get("f1", 0), 4),
    "sp_em":               round(metrics.get("sp_em", 0), 4),
    "sp_f1":               round(metrics.get("sp_f1", 0), 4),
    "sp_prec":             round(metrics.get("sp_prec", 0), 4),
    "sp_recall":           round(metrics.get("sp_recall", 0), 4),
    "joint_em":            round(metrics.get("joint_em", 0), 4),
    "joint_f1":            round(metrics.get("joint_f1", 0), 4),
    "RAGAS Faithfulness":  round(metrics.get("faithfulness", 0), 4),
    "MRR":                 round(metrics.get("mrr", 0), 4),
    "Latency Mean (s)":    round(metrics.get("latency_mean_s", 0), 4),
    "Latency Median (s)":  round(metrics.get("latency_median_s", 0), 4),
    "Latency p95 (s)":     round(metrics.get("latency_p95_s", 0), 4),
    "Cost Proxy Mean":     round(metrics.get("cost_proxy_mean", 0), 4),
    "Cost Proxy Total":    round(metrics.get("cost_proxy_total", 0), 4),
}

header = list(row.keys())
write_header = not os.path.exists(CSV_PATH)

with open(CSV_PATH, "a", newline="") as f:
    w = csv.DictWriter(f, fieldnames=header)
    if write_header:
        w.writeheader()
    w.writerow(row)

print(f"Appended metrics row to {CSV_PATH}")
print("  ", row)

## Step 13 — Inspect Multi-Stage Predictions & Key Usage

In [ ]:
for d in details[:5]:
    print(f"Q: {d['question']}")
    print(f"  Gold: {d['gold_answer']} | Pred: {d['pred_answer']}")
    for sl in d['stage_log']:
        print(f"  Stage {sl['stage']}: query='{sl['query'][:80]}...'")
    print(f"  Final: {[(r['title'], r['sent_id']) for r in d['retrieved_context']]}\n")

print("--- API Key Usage ---")
key_manager.status()